In [0]:
create external location if not exists streaminbound 
url 'abfss://fakerstream@dbstorageext01.dfs.core.windows.net' 
with (credential faker_cred)

In [0]:
create catalog if not exists filestreaming
comment "This is catalog holds faker stream source  event='File'"

In [0]:
create schema if not exists filestreaming.fakerstream
comment "This is schema holds faker stream source  event='File'"


In [0]:
create external volume if not exists filestreaming.fakerstream.inbound
location 'abfss://fakerstream@dbstorageext01.dfs.core.windows.net/inboundfiles/'
comment 'Landing zone for faker stream  files';

In [0]:
%python
display(
  dbutils.fs.ls("/Volumes/filestreaming/fakerstream/inbound")
)

In [0]:
%python
!pip install faker

In [0]:
%python
from faker import Faker
import json, time, random
from datetime import datetime

fake = Faker()

while True:
    event = {
        "event_id": fake.uuid4(),  # unique tracking
        "order_id": random.randint(1, 10000),
        "amount": round(random.uniform(10, 500), 2),

        # 🔥 IMPORTANT TIMESTAMPS
        "event_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "producer_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    file_name = f"/Volumes/filestreaming/fakerstream/inbound/{int(time.time()*1000)}.json"

    with open(file_name, "w") as f:
        json.dump(event, f)

    time.sleep(2)